> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# HS project 05 — genomic / matrix experts and fast nonlinear prediction

The public expert baseline is the genetic mean: training trait mean plus mvBLUP breeding value. Rejected experts return that baseline, including for partial rows. It differs from mvPredict’s full conditional phenotype posterior. Callers providing `blup=` must supply this phenotype-scale genetic mean; omitting it lets the package compute it.

In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__

In [ ]:
train=pd.read_csv(TRAIN_CSV,index_col=0);train.index=train.index.astype(str);train.columns=train.columns.str.replace('regressedlr_','',regex=False)
test_rows=pd.read_csv(TEST_ROWS_CSV,index_col=0);test_rows.index=test_rows.index.astype(str);test_rows.columns=test_rows.columns.str.replace('regressedlr_','',regex=False);test_rows=test_rows.reindex(index=train.index,columns=train.columns)
test_elems=pd.read_csv(TEST_ELEMS_CSV,index_col=0);test_elems.index=test_elems.index.astype(str);test_elems.columns=test_elems.columns.str.replace('regressedlr_','',regex=False);test_elems=test_elems.reindex(index=train.index,columns=train.columns)
store=vigor.open_store(ANALYSIS_STORE,mode='a',lazy=False)
Ga=np.asarray(vigor.open_grm(store,component='additive'),float);Gd=np.asarray(vigor.open_grm(store,component='dominance'),float)
fit_mask=train.notna().any(axis=1).to_numpy()
features_pc=vigor.make_genomic_features({'additive':Ga,'dominance':Gd},fit_mask=fit_mask,n_grm_components=32,index=train.index,random_state=SEED)["features"].to_pandas()
features_snp=vigor.genomic_features_from_plink1(
    PLINK_PREFIX,sample_ids=train.index,fit_mask=fit_mask,max_variants=GENOMIC_EXPERT_MAX_VARIANTS,
    strategy='equidistant',include_dominance=True,random_state=SEED)
genomic_features=pd.concat([features_pc,features_snp],axis=1)
store.write_expert_features(genomic_features,group='experts/candidate_features',mode='a')
print('Cached expert feature matrix:',genomic_features.shape)


In [ ]:
# Load the persisted GE-full covariance/prediction from notebooks 02/04.
cov=store.open_dataset('covariance/GE_full',lazy=False)
G=np.asarray(cov['G'],float); E=np.asarray(cov['E'],float)
if G.ndim==3:G=G.sum(axis=0)
try:
    pds=store.open_dataset('prediction/GE_full',lazy=False)
    base=pd.DataFrame(np.asarray(pds['mvblup'],float),index=train.index,columns=train.columns)
except Exception:
    base=pd.DataFrame(np.asarray(vigor.mvBLUP(train,Ga,G,E=E,missing='exact',solver='pcg',solver_tol=MVBLUP_TOL,
        solver_maxiter=MVBLUP_MAXITER,preconditioner=MVBLUP_PRECONDITIONER,accuracy='exact',blas_threads=BLAS_THREADS),float),index=train.index,columns=train.columns)

# Public expert baseline is mean + breeding value on phenotype units.
base=base.add(train.mean(axis=0),axis=1)

cov_kwargs=dict(
    rank_G=FULL_RANK_G,rank_E=FULL_RANK_E,solver=FULL_SOLVER,
    variance_closure=VARIANCE_CLOSURE,variance_closure_strength=VARIANCE_CLOSURE_STRENGTH,
    analytic_se=False,n_boot=0,random_state=SEED,blas_threads=BLAS_THREADS,progress=False,
)


In [ ]:
# Compare unseen-genomic strategies on exactly the same folds/features.
# Matrix methods are evaluated simultaneously for partially phenotyped animals.
strategies={
    'screened_ridge':dict(feature_methods=(),genomic_strategy='screened_ridge'),
    'elasticnet':dict(feature_methods=('elasticnet',),genomic_strategy='per_trait'),
    'lightgbm_selected':dict(feature_methods=('lightgbm_selected',),genomic_strategy='per_trait'),
}
fits={};predictions={};rows=[]

def _score(truth,pred):
    vals=[]
    for t in train.columns:
        y=truth[t].to_numpy(float);p=pred[t].to_numpy(float);m=np.isfinite(y)&np.isfinite(p)
        if m.sum()<MIN_R2_N or np.std(y[m])==0 or np.std(p[m])==0: continue
        vals.append(np.corrcoef(y[m],p[m])[0,1]**2)
    return float(np.nanmedian(vals)) if vals else np.nan

for name,opt in strategies.items():
    fit=vigor.fit_blup_prediction_experts(
        train,Ga,G,E=E,blup=base,genomic_features=genomic_features,
        feature_methods=opt['feature_methods'],matrix_methods=MATRIX_METHODS,
        n_splits=EXPERT_N_SPLITS,random_state=SEED,crossfit_refit_covariance=True,
        covariance_kwargs=cov_kwargs,selection_cv=EXPERT_SELECTION_CV,
        genomic_strategy=opt['genomic_strategy'],genomic_screen_top_k=GENOMIC_EXPERT_TOP_K,
        genomic_screen_alpha=GENOMIC_EXPERT_ALPHA,genomic_screen_min_gain=.002,genomic_screen_min_z=1.96,
        min_matrix_gain=.002,matrix_calibrate_oof=True,matrix_calibration_min_points=8,
        matrix_calibration_max_scale=1.0,
    )
    predout=vigor.predict_blup_prediction_experts(fit,train,blup=base,GRM=Ga,genomic_features=genomic_features)
    pred=predout['prediction'].to_pandas(); fits[name]=fit; predictions[name]=pred
    store.write_predictions(pred,name='prediction',group=f'prediction/expert_{name}',mode='a')
    rows.append({'method':name,'unseen_mode':fit.attrs['unseen_mode'],'partial_mode':fit.attrs['partial_mode'],
                 'unseen_median_r2':_score(test_rows,pred),'partial_median_r2':_score(test_elems,pred)})

expert_summary=pd.DataFrame(rows).sort_values('unseen_median_r2',ascending=False)
expert_summary.to_csv(RESULTS/'expert_method_comparison.csv',index=False)
display(expert_summary)
print('Base unseen median R²:',_score(test_rows,base))
print('Base partial median R²:',_score(test_elems,base))

# SHAP is deliberately deferred to notebook 07 after selecting the best nonlinear model.
# TreeSHAP is predictive attribution, not LD-aware fine mapping; notebook 07 aggregates
# SNP attribution over empirical LD groups before interpretation.


Feature selection is **fold-local**, not random. The initial 8,192-SNP panel is phenotype-independent and genome-spaced; supervised top-k ranking occurs only inside training folds. After establishing a large prediction gain, explain LightGBM with OOF SHAP and aggregate SNP attributions over LD blocks.